Color Segmentation using Gaussian Mixture Models — Overview and How to Run

Approach
- Convert images to HSV color space to better separate hue/saturation of the orange ball from background.
- Build a single-Gaussian baseline: extract candidate orange pixels from training images using an HSV mask, then estimate mean and covariance by maximum likelihood.
- Implement a K-component Gaussian Mixture Model (GMM) from scratch (no sklearn): random initialization, EM iterations (E-step responsibilities, M-step updates for means, covariances with small jitter for stability, and mixture weights) until convergence or max iterations.
- Segment test images by evaluating per-pixel likelihood under the learned mixture and thresholding a posterior-proportional score (likelihood × prior).
- Visualize learned components as 3D ellipsoids in HSV space and show side-by-side segmentation results.

Assumptions and notes
- Images contain an orange soccer ball under similar lighting; HSV is used and channels may be normalized during modeling.
- Pixels are treated independently for modeling; a small prior and numerical jitter are used to avoid singular covariances.
- All code is implemented from scratch for educational clarity; thresholds and K are tunable.

Run locally
- Install dependencies: from the repository root, create a virtual environment and pip install -r requirements.txt.
- Prepare data directories:
  - projects/color-segmentation-gmm/data/train_images/*.jpg
  - projects/color-segmentation-gmm/data/test_images/*.jpg
  Or set environment variables:
  - COLOR_SEG_DATA_DIR to a directory containing train_images/ and test_images/
  - COLOR_SEG_RESULTS_DIR to choose where outputs are written (default: projects/color-segmentation-gmm/results)
- Open this notebook and:
  - Use the Single-Gaussian section to estimate a baseline model.
  - Use the GMM section to train a K-component model (set mode_flag = 0 to train; mode_flag = 1 to run testing/visualization with existing parameters).
- The notebook does not download data automatically.

# Color Segmentation using Gaussian Mixture Models

# Introduction

Have you ever played with these adorable Nao robots? Click [here](http://www.youtube.com/watch?feature=player_embedded&v=Gy_wbhQxd_0) to watch a cool demo.

Nao robots are star players in RoboCup, an annual autonomous robot soccer competitions. Would you like to help us in Nao’s soccer training? We need to train Nao to detect a soccer ball and estimate the depth of the ball to know how far to kick.

Nao’s training has two phases:

- Color Segmentation using Gaussian Mixture Model (GMM)
- Ball Distance Estimation

## Data and Local Setup

This notebook expects a local dataset of training and test images of an orange soccer ball. It does not download data automatically.

How to run locally
- Install dependencies: from the repository root, create a virtual environment and run `pip install -r requirements.txt`.
- Place images under the following directories:
  - `projects/color-segmentation-gmm/data/train_images/*.jpg`
  - `projects/color-segmentation-gmm/data/test_images/*.jpg`
- Alternatively, set environment variables to point to your data/output locations:
  - `COLOR_SEG_DATA_DIR` → directory containing `train_images/` and `test_images/` subfolders
  - `COLOR_SEG_RESULTS_DIR` → directory to write saved figures/masks (defaults to `projects/color-segmentation-gmm/results`)

The next cell parameterizes these paths so the rest of the notebook uses repository-relative paths or your environment overrides.

In [ ]:
# Configure data and results directories (override with environment variables if needed)
import os

# Base data directory expected to contain 'train_images/' and 'test_images/' subfolders
DATA_DIR = os.getenv("COLOR_SEG_DATA_DIR", os.path.join("projects", "color-segmentation-gmm", "data"))
TRAIN_DIR = os.path.join(DATA_DIR, "train_images")
TEST_DIR = os.path.join(DATA_DIR, "test_images")

# Output directory for any saved artifacts or visualizations
RESULTS_DIR = os.getenv("COLOR_SEG_RESULTS_DIR", os.path.join("projects", "color-segmentation-gmm", "results"))
os.makedirs(RESULTS_DIR, exist_ok=True)

print("TRAIN_DIR =", TRAIN_DIR)
print("TEST_DIR  =", TEST_DIR)
print("RESULTS_DIR =", RESULTS_DIR)

In [ ]:
# Check whether the training images are available locally
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import os, glob

sample_images = sorted(glob.glob(os.path.join(TRAIN_DIR, "*.jpg")))
if sample_images:
    train_image = mpimg.imread(sample_images[0])
    plt.imshow(train_image)
    plt.axis("off")
    plt.show()
else:
    print(f"No images found in {TRAIN_DIR}. See the Data and Local Setup instructions above.")

In [ ]:
# Imports
import cv2
import os
import glob
import numpy as np

# Read in training images
path = TRAIN_DIR
files = "*.jpg"

file_path = glob.glob(os.path.join(path, files))
print(file_path)
total = []

# Iterate over training images to extract orange pixels using masks
for x in file_path:
    image = cv2.imread(x)  # read image x
    image_array = np.array(image)  # convert the image into an np array
    # lower = np.array([10, 100, 20]) #a BGR lowerbound, this is a dark green
    # upper = np.array([25, 255, 255]) #a BGR upperbound, this is a bright yellow

    # mostly just the ball (ONLY WORKS WHEN WE ASSUME IMAGE IS RGB, BUT IT
    # ACTUALLY COMES IN AS BGR)
    lower = np.array([110, 100, 100])
    upper = np.array([120, 255, 255])

    # trying to improve on last one

    hsv = cv2.cvtColor(
        image_array, cv2.COLOR_RGB2HSV
    )  # create new copy of image w/ HSV color space
    mask = cv2.inRange(
        hsv, lower, upper
    )  # grab the subset of the image who's color lies within the given bound
    # Question about previous line: does mask just tell you which pixels fall in the specified range?
    # Answer - The mask will use the lower and upper bounds given by the array of color values and matches pixels based on those ranges.

    # Ensure results directory exists
    os.makedirs(RESULTS_DIR, exist_ok=True)

    # write original image to disk
    cv2.imwrite(os.path.join(RESULTS_DIR, "x.jpg"), image)
    # write the mask to disk
    cv2.imwrite(os.path.join(RESULTS_DIR, "mask.jpg"), mask)

    # write hsv
    cv2.imwrite(os.path.join(RESULTS_DIR, "hsv.jpg"), cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB))

    # write image_masked.jpg
    image_masked = cv2.bitwise_and(image, image, mask=mask)
    cv2.imwrite(os.path.join(RESULTS_DIR, "image_masked.jpg"), image_masked)

    orange_total = hsv[mask > 0]
    total.extend(orange_total)


total = np.array(total)
print(total)

# Compute mean and covariance using MLE(Maximum Likelihood Estimation)
mean = np.mean(total, axis=0)
print("mean: ")
print(mean)
covariance = np.cov(total.T)
# covariance = np.cov(total)

print(covariance)
# Compute PDF(Probability Density Function) of single gaussian model


def pdf(x, mean, covariance):
    diff = (x - mean).reshape(3, 1)  # (X - u) part of the equatation

    exponent = -0.5 * (diff.transpose() @ np.linalg.inv(covariance) @ diff)
    numerator = (np.exp(exponent)).item()
    denominator = np.sqrt(((2 * np.pi) ** 3) * abs(np.linalg.det(covariance)))

    pdf = numerator / denominator

    return pdf


# Set parameters (threshold, prior)
threshhold = 0.00000000000001


# P(Orange)
prior = 0.000000001


# Send test images into algorithm to detect orange ball
test_path = TEST_DIR
test_files = "*.jpg"
test_file_path = glob.glob(os.path.join(test_path, test_files))

outputs = []

i = 0
for x in test_file_path:
    image = cv2.imread(x)
    image = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)

    originalImage = image.copy()

    for row in image:
        for pixel in row:

            # pass pixel into PDF to get P(x|Orange)
            likelihood = pdf(pixel, mean, covariance)

            # assumption: P(Orange|x) is proportional to P(x|Orange)*P(Orange)
            posterior = likelihood * prior

            if posterior >= threshhold:
                pixel[0] = 255
                pixel[1] = 255
                pixel[2] = 255
            else:
                pixel[0] = 0
                pixel[1] = 0
                pixel[2] = 0

    outputs.append((originalImage, image))
    # cv2.imwrite(f"singleGaussianOutput_{x.replace('/', '_')}.jpg", image)
    # print(x)
    # i += 1

    # plt.imshow(image)
    # plt.axis("off")
    # plt.show()

In [ ]:
# Show you result here

# TODO: change code to show original and output next to each other
# for image in outputs:
#   plt.imshow(image)
#   plt.axis("off")
#   plt.show()


for original, result in outputs:
    fig, axes = plt.subplots(1, 2, figsize=(24, 8))

    original_rbg = cv2.cvtColor(original, cv2.COLOR_HSV2BGR)
    # result_rbg = cv2.cvtColor(result, cv2.COLOR_HSV2RGB)
    # axes[0].imshow(cv2.cvtColor(original, cv2.COLOR_HSV2RGB))
    axes[0].imshow(original_rbg)
    axes[0].set_title("Original")

    axes[1].imshow(result)
    axes[1].set_title("Model Output")

    plt.tight_layout()
    plt.show()

In [ ]:
# K = number of gaussians to train
def trainGMM(K, pixels):
    print(f"Pixels in trainGMM: {pixels}")

    # convert pixels to int
    pixels = pixels.astype(np.float128)

    # Set convergence criteria and initialize scaling factor, gaussian mean and covariance
    pixels_length = pixels.shape[0]

    # we need this to be an array of k elements to compare with mean difference
    # change fill_value to change threshold
    threshold = np.full(
        shape=(K, 3), fill_value=0.00001
    )

    randRange = 1000

    weights = np.random.rand(K) * randRange
    weights = weights / weights.sum()
    means = np.empty((K, 3))

    covariances = np.empty((K, 3, 3))

    for i in range(K):

        pixel1 = pixels[np.random.randint(0, pixels_length)]
        pixel2 = pixels[np.random.randint(0, pixels_length)]
        pixel3 = pixels[np.random.randint(0, pixels_length)]

        print(f"pixel1.dtype: {pixel1.dtype}")

        # mean is avg of 3 pixels chosen at random
        # means[i] = (pixel1/3 + pixel2/3 + pixel3/3)
        means[i] = np.random.rand(3)

        # covariance matrix is identiy matrix
        covariances[i] = np.identity(3) * (np.random.rand(3) + 10**-300)

    # list of 3x3 elements
    print(f"covariance[0]: {covariances[0]}")
    print(f"means[0]: {means[0]}")
    print(f"weights: {weights}")

    maxIterations = 100

    oldMeans = means.copy() - 1

    a = np.zeros(
        (pixels_length, K)
    )

    iter = 0

    while not (
        iter > maxIterations or (abs(oldMeans - means) <= threshold).all()
    ):

        # Main training algorithm (EM algorithm)

        print(f"shape of a {a.shape}")
        print(f"a[0][0]: {a[0][0]}")
        # expectation step

        expectationDenominator = np.zeros((len(pixels)))

        for i, pixel in enumerate(pixels):
            for j in range(0, K):
                expectationDenominator[i] += weights[j] * pdf(
                    pixel, means[j], covariances[j]
                )

        aLocationsSet = 0
        # iterate over models
        for j in range(0, K):
            # iterate over ALL pixels in images
            for i, pixel in enumerate(pixels):
                aLocationsSet += 1
                a[i][j] = (
                    weights[j] * pdf(pixel, means[j], covariances[j])
                ) / expectationDenominator[i]

        print(f"a: {a}")
        print(f"aLocationsSet: {aLocationsSet}")

        # maximization step
        for j in range(0, K):

            newMeanTop = np.zeros((1, 3))
            newMeanBottom = 0

            newVarianceTop = np.zeros((1, 3, 3))
            newVarianceBottom = 0

            newWeight = 0

            for i in range(0, pixels_length):

                newMeanTop += a[i][j] * pixels[i]
                newMeanBottom += a[i][j]

                pixelMinusMean = (pixels[i] - means[j]).reshape((3, 1))

                newVarianceTop += a[i][j] * np.dot(pixelMinusMean, pixelMinusMean.T)
                newVarianceBottom += a[i][j]

                newWeight += a[i][j]

            print(f"newMeanTop: {newMeanTop}")
            print(f"newMeanBottom: {newMeanBottom}")
            print(f"newVarianceTop: {newVarianceTop}")
            print(f"newVarianceBottom: {newVarianceBottom}")
            newMean = newMeanTop / newMeanBottom
            newVariance = (newVarianceTop / newVarianceBottom) + (
                np.identity(3) * (np.random.rand(3) * 1e-6)
            )
            newWeight = newWeight / pixels_length

            oldMeans[j] = means[j]
            means[j] = newMean
            weights[j] = newWeight
            covariances[j] = newVariance

        iter += 1

        print(f"iteration number: {iter}")
        print(f"oldMeans: {oldMeans}")
        print(f"means (new): {means}")
        print(
            f"(abs(oldMeans - means) <= threshold): {(abs(oldMeans - means) <= threshold)}"
        )
        print(f"weights (new): {weights}")
        print(f"covariances (neW): {covariances}")

    return weights, means, covariances


def testGMM(Model_parameters, threshold, prior):
    K, scalars, means, covariances = Model_parameters
    # Read test images
    path = TEST_DIR
    files = "*.jpg"
    results = RESULTS_DIR
    os.makedirs(results, exist_ok=True)
    # Load test images
    file_path = glob.glob(os.path.join(path, files))
    pixel_values = []
    locations = []
    posteriors = []

    # Main testing loop over all test images and use thresholding to get the orange ball
    for x in file_path:  # CHRIS
        image = cv2.imread(x)
        image = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)  # CHRIS
        image_pixels = []
        image_locations = []
        image_posteriors = []  # FOR QUICKER TESTING

        for i, row in enumerate(image):
            for j, pixel in enumerate(row):
                posterior = 0

                for cluster in range(K):  # Summation from article
                    posterior += scalars[cluster] * pdf(
                        pixel / np.array([179, 255, 255]),
                        means[cluster],
                        covariances[cluster],
                    )

                # if posterior >= threshold:
                if True:
                    image_pixels.append(pixel)
                    image_locations.append((i, j))
                    image_posteriors.append(posterior)

        pixel_values.append(image_pixels)
        locations.append(image_locations)
        posteriors.append(image_posteriors)  # FOR QUICKER TESTING

    # Saving predictions to the result folder (optional, uncomment if writing out)
    # results_file = os.path.join(results, os.path.basename(x))
    # cv2.imwrite(results_file,image)

    return (pixel_values, locations, posteriors)


def measureDepth(cluster_parameters):
    pass


# Identify the pixels which belong to the orange ball
# (Experiment with cv2 morphological operators like open/close to refine masks.)


def plotGMM(Model_parameters, distance, pixels):
    K, scaling, mean, covariance = Model_parameters
    print(f"K in plotGMM: {K}")

    fig = plt.figure()
    axis = fig.add_subplot(111, projection="3d")

    colors = ["r", "g", "b"]

    bias = 0

    for i in range(0, K):
        print(f"scaling[{i}] in plotGMM: {scaling[i]}")
        print(f"mean[{i}] in plotGMM: {mean[i]}")
        print(f"covariance[{i}] in plotGMM: {covariance[i]}")
        u = np.linspace(0, 2 * np.pi, 100)
        v = np.linspace(0, np.pi, 100)

        x = np.outer(np.cos(u), np.sin(v))
        y = np.outer(np.sin(u), np.sin(v))
        z = np.outer(np.ones_like(u), np.cos(v))
        sphere = np.stack((x, y, z), axis=-1)[..., None]

        e, v = np.linalg.eig(covariance[i])
        s = v @ np.diag(np.sqrt(e)) @ v.T

        ellipsoid = (s @ sphere).squeeze(-1) + mean[i]

        axis.plot_surface(
            *ellipsoid.transpose(2, 0, 1),
            rstride=4,
            cstride=4,
            color=(219 / 255, 101 / 255, 65 / 255, scaling[i]),
        )

    axis.set_xlabel("X")
    axis.set_ylabel("Y")
    axis.set_zlabel("Z")
    plt.show()

    fig = plt.figure()
    axis = fig.add_subplot(111, projection="3d")

    pixels = pixels.transpose()

    axis.scatter(pixels[0], pixels[1], pixels[2], c=(0, 0, 0, 0.5), s=10)

    axis.set_xlabel("X")
    axis.set_ylabel("Y")
    axis.set_zlabel("Z")
    plt.show()

In [ ]:
scalars, means, covariances = 0, 0, 0  # Initialize GMM parameters

In [ ]:
# Main driver: train a GMM on orange-ball pixels or test/visualize with existing parameters

threshold = 5e-8  # Threshold for clustering (tunable)
K = 4  # Number of Gaussians (tunable)

# Set to 0 for training (default), set to 1 for testing/visualization with existing parameters
mode_flag = 0

if mode_flag == 0:

    # Read in training images
    path = TRAIN_DIR
    files = "*.jpg"

    file_path = glob.glob(os.path.join(path, files))

    # Iterate over training images to extract orange pixels using masks
    pixels = []

    for x in file_path:  # Looping through images
        image = cv2.imread(x)
        hsv_image = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)  # CHRIS

        # Setting bounds for hsv mask
        lower_orange = np.array([110, 100, 100])
        upper_orange = np.array([120, 255, 255])
        mask = cv2.inRange(hsv_image, lower_orange, upper_orange)
        # Masking Images
        orange_pixels = hsv_image[mask > 0]
        pixels.extend(orange_pixels)

    pixels = np.array(pixels) / np.array([179, 255, 255])

    print(f"pixels.shape going into trainGMM: {pixels.shape}")  # CHRIS
    print(f"pixels: {pixels}")

    scalars, means, covariances = trainGMM(K, pixels)


else:
    # Load Model (expected to be defined in prior cells)
    print(scalars)
    print(means)
    print(covariances)

    cluster_parameters = testGMM((K, scalars, means, covariances), threshold, prior)
    # distance = measureDepth(cluster_parameters)
    plotGMM((K, scalars, means, covariances), 1, pixels)

In [ ]:
# for x in file_path[-1:]: # Looping through images
#   image = cv2.imread(x)
#   print(image.shape)

# print(image.shape)
# print(image.dtype)
pixel_value_list, pixel_location_list, posteriors_list = cluster_parameters
# print(pixel_value_list[0])
# print(len(pixel_value_list))
# print(pixel_location_list[0])
# print(len(pixel_location_list))

# for image_num in range(len(pixel_value_list)):
#   for row_num in range(len(pixel_value_list[image_num])):
#     for col_num in range(len(pixel_value_list[image_num][row_num])):
#       image[image_num][pixel_location_list[row_num][col_num][0]][pixel_location_list[row_num][col_num][1]] = pixel_value_list[row_num][col_num]


threshold = 1.9e-1

for i in range(0, 8):
    image = np.zeros((640, 640, 3), dtype=np.uint8)
    fig, axes = plt.subplots(1, 1, figsize=(24, 8))

    print(i)
    for value, location, posterior in zip(
        pixel_value_list[i], pixel_location_list[i], posteriors_list[i]
    ):
        # print(location)
        if posterior >= threshold:
            image[location[0]][location[1]] = value

        # if (value > max).any:
        #   max = value
        # if (value < min).any:
        #   min = value

        # print(image.shape)
        # print(image.dtype)

        # print(max)
        # print(min)
    image = cv2.cvtColor(image, cv2.COLOR_HSV2BGR)
    # image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    axes.imshow(image)

In [ ]:
pixels, locations = cluster_parameters
print(pixels)
print(locations)